# Stage 05 — Diagnostics

Run all 12 automated checks from `skills/stage_05.md` and write
`data/results/diagnostics_flags.json`.

In [1]:
from paths import *
import json
import numpy as np

# ── Load all result files ──────────────────────────────────────────
spec      = json.loads(PAPER_SPEC.read_text())
rep_check = json.loads(REPLICATION_CHECK.read_text())
rep_res   = json.loads(REPLICATION_RESULTS.read_text())
dml_res   = json.loads(DML_RESULTS.read_text())

# Optional files
hte_path = RESULTS_DIR / "hte_results.json"
hte_res  = json.loads(hte_path.read_text()) if hte_path.exists() else None

cf_path  = RESULTS_DIR / "causal_forest_results.json"
cf_res   = json.loads(cf_path.read_text()) if cf_path.exists() else None

# ── Primary specification values ───────────────────────────────────
primary   = dml_res["specifications"][0]
pub_coef  = spec["main_results"][0]["coef"]
dml_coef  = primary["preferred_coef"]
dml_lo    = primary["preferred_ci_lo"]
dml_hi    = primary["preferred_ci_hi"]
dml_se    = primary["learners"][primary["preferred_learner"]]["se"]
n_obs     = primary["n_obs"]

print(f"Published main coef : {pub_coef}")
print(f"DML preferred coef  : {dml_coef:.6f}  CI: [{dml_lo:.6f}, {dml_hi:.6f}]")
print(f"DML SE              : {dml_se:.6f}")
print(f"N obs               : {n_obs}")
print(f"HTE available       : {hte_res is not None}")
print(f"CF  available       : {cf_res is not None}")

Published main coef : 0.035
DML preferred coef  : 0.019718  CI: [-0.004520, 0.043956]
DML SE              : 0.012366
N obs               : 63
HTE available       : True
CF  available       : False


In [2]:
# ── Helper ─────────────────────────────────────────────────────────
def flag(status, value, note):
    return {"status": status, "value": value, "note": note}

checks = {}

# ===================================================================
# Check 1: Replication pass
# ===================================================================
worst = rep_check["worst_rel_diff_pct"]
if worst < 5:
    s1 = "PASS"
elif worst < 15:
    s1 = "WARN"
else:
    s1 = "FAIL"
checks["replication_pass"] = flag(s1, worst, f"{worst:.2f}% max deviation")
print(f"Check  1  replication_pass      : {s1}  ({worst:.2f}%)")

# ===================================================================
# Check 2: DML coefficient direction
# ===================================================================
same_sign = (pub_coef * dml_coef) > 0
rel_mag   = abs(dml_coef - pub_coef) / abs(pub_coef) if pub_coef != 0 else float("nan")

if same_sign and rel_mag <= 0.5:
    s2 = "PASS"
elif same_sign:
    s2 = "WARN"
else:
    s2 = "FAIL"
note2 = f"same sign, {rel_mag*100:.1f}% magnitude diff" if same_sign else "SIGN CHANGE"
checks["dml_direction"] = flag(s2, round(dml_coef, 6), note2)
print(f"Check  2  dml_direction         : {s2}  ({note2})")

# ===================================================================
# Check 3: DML CI coverage
# ===================================================================
inside_ci = dml_lo <= pub_coef <= dml_hi
if inside_ci:
    s3 = "PASS"
    note3 = "published coef inside DML CI"
else:
    # distance in SE units
    if pub_coef < dml_lo:
        dist_se = (dml_lo - pub_coef) / dml_se if dml_se > 0 else float("inf")
    else:
        dist_se = (pub_coef - dml_hi) / dml_se if dml_se > 0 else float("inf")
    if dist_se <= 1:
        s3 = "WARN"
        note3 = f"published coef outside DML CI by {dist_se:.2f} SE"
    else:
        s3 = "FAIL"
        note3 = f"published coef >{dist_se:.1f} SE outside DML CI"
checks["dml_ci_coverage"] = flag(s3, round(pub_coef, 6), note3)
print(f"Check  3  dml_ci_coverage       : {s3}  ({note3})")

# ===================================================================
# Check 4: Nuisance model quality
# ===================================================================
best_learner = primary["learners"][primary["preferred_learner"]]
r2_out = best_learner["nuisance"]["r2_outcome"]
r2_trt = best_learner["nuisance"]["r2_treatment"]

if r2_out > 0.3 and r2_trt > 0.3:
    s4 = "PASS"
elif r2_out >= 0.1 and r2_trt >= 0.1:
    s4 = "WARN"
else:
    s4 = "FAIL"
note4 = f"r2_outcome={r2_out:.3f}, r2_treatment={r2_trt:.3f}"
checks["nuisance_quality"] = flag(s4, {"r2_outcome": round(r2_out, 4), "r2_treatment": round(r2_trt, 4)}, note4)
print(f"Check  4  nuisance_quality      : {s4}  ({note4})")

# ===================================================================
# Check 5: Sample size
# ===================================================================
if n_obs >= 50:
    s5 = "PASS"
elif n_obs >= 30:
    s5 = "WARN"
else:
    s5 = "FAIL"
checks["sample_size"] = flag(s5, n_obs, f"n={n_obs}")
print(f"Check  5  sample_size           : {s5}  (n={n_obs})")

# ===================================================================
# Check 6: HTE heterogeneity signal
# ===================================================================
if hte_res is not None:
    het_detected = hte_res.get("gate", {}).get("heterogeneity_detected", False)
    if het_detected:
        s6 = "PASS"
        note6 = "heterogeneity detected (non-overlapping GATE CIs)"
    else:
        s6 = "WARN"
        note6 = "all GATE CIs overlap -- homogeneous effect"
    checks["hte_heterogeneity"] = flag(s6, het_detected, note6)
    print(f"Check  6  hte_heterogeneity     : {s6}  ({note6})")
else:
    checks["hte_heterogeneity"] = flag("SKIP", None, "hte_results.json not found")
    print(f"Check  6  hte_heterogeneity     : SKIP")

# ===================================================================
# Check 7: Causal forest ATE consistency
# ===================================================================
if cf_res is not None:
    cf_ate = cf_res.get("ate", cf_res.get("ate_mean"))
    if dml_coef is not None:  # DML+CF pipeline
        cf_same_sign = (cf_ate * dml_coef) > 0
        cf_mag_diff  = abs(cf_ate - dml_coef) / abs(dml_coef) if dml_coef != 0 else float("nan")
        if cf_same_sign and cf_mag_diff < 0.5:
            s7 = "PASS"
        elif cf_same_sign:
            s7 = "WARN"
        else:
            s7 = "FAIL"
        note7 = f"cf_ate={cf_ate:.4f}, {cf_mag_diff*100:.1f}% diff vs DML" if cf_same_sign else "sign change vs DML"
    else:  # CF-only pipeline
        cf_same_sign = (cf_ate * pub_coef) > 0
        cf_mag_diff  = abs(cf_ate - pub_coef) / abs(pub_coef) if pub_coef != 0 else float("nan")
        if cf_same_sign:
            s7 = "PASS"
        elif cf_mag_diff > 0.5:
            s7 = "WARN"
        else:
            s7 = "FAIL"
        note7 = f"cf_ate={cf_ate:.4f} vs published"
    checks["cf_ate_consistency"] = flag(s7, round(cf_ate, 6), note7)
    print(f"Check  7  cf_ate_consistency    : {s7}  ({note7})")
else:
    checks["cf_ate_consistency"] = flag("SKIP", None, "causal_forest_results.json not found")
    print(f"Check  7  cf_ate_consistency    : SKIP  (no CF)")

# ===================================================================
# Check 8: Causal forest CATE plausibility
# ===================================================================
if cf_res is not None:
    pct_sig = cf_res.get("pct_significant", None)
    if pct_sig is not None:
        if 0.10 < pct_sig < 0.95:
            s8 = "PASS"
        elif pct_sig > 0.99 and n_obs < 200:
            s8 = "FAIL"
        else:
            s8 = "WARN"
        note8 = f"{pct_sig*100:.1f}% of individual CATEs significant"
        checks["cf_cate_plausibility"] = flag(s8, round(pct_sig, 4), note8)
        print(f"Check  8  cf_cate_plausibility  : {s8}  ({note8})")
    else:
        checks["cf_cate_plausibility"] = flag("SKIP", None, "pct_significant not in CF results")
        print(f"Check  8  cf_cate_plausibility  : SKIP")
else:
    checks["cf_cate_plausibility"] = flag("SKIP", None, "causal_forest_results.json not found")
    print(f"Check  8  cf_cate_plausibility  : SKIP  (no CF)")

# ===================================================================
# Check 9: ATE SE plausibility (CF + HTE)
# ===================================================================
if cf_res is not None and hte_res is not None:
    ate_ci_lo = cf_res.get("ate_ci_lo")
    ate_ci_hi = cf_res.get("ate_ci_hi")
    gate_groups = hte_res.get("gate", {}).get("groups", [])
    if ate_ci_lo is not None and ate_ci_hi is not None and len(gate_groups) > 0:
        ate_ci_width = ate_ci_hi - ate_ci_lo
        gate_widths  = [g["ci_hi"] - g["ci_lo"] for g in gate_groups]
        mean_gate_w  = np.mean(gate_widths)
        ratio = mean_gate_w / ate_ci_width if ate_ci_width > 0 else float("inf")
        if ratio < 5:
            s9 = "PASS"
        elif ratio < 10:
            s9 = "WARN"
        else:
            s9 = "FAIL"
        note9 = f"GATE/ATE CI width ratio = {ratio:.2f}"
        checks["ate_se_plausibility"] = flag(s9, round(ratio, 3), note9)
        print(f"Check  9  ate_se_plausibility   : {s9}  ({note9})")
    else:
        checks["ate_se_plausibility"] = flag("SKIP", None, "missing CI bounds in CF or GATE groups")
        print(f"Check  9  ate_se_plausibility   : SKIP")
else:
    checks["ate_se_plausibility"] = flag("SKIP", None, "requires both CF and HTE results")
    print(f"Check  9  ate_se_plausibility   : SKIP  (no CF)")

# ===================================================================
# Check 10: Cross-fitting stability
# ===================================================================
# Look for per_rep_coefs / per_rep_ses in the Best learner
best_data   = primary["learners"].get("Best", primary["learners"].get(primary["preferred_learner"], {}))
per_rep_coefs = best_data.get("per_rep_coefs")
per_rep_ses   = best_data.get("per_rep_ses")

if per_rep_coefs is not None and per_rep_ses is not None and len(per_rep_coefs) > 1:
    sd_reps   = float(np.std(per_rep_coefs, ddof=1))
    median_se = float(np.median(per_rep_ses))
    ratio10   = sd_reps / median_se if median_se > 0 else float("inf")
    if ratio10 < 0.5:
        s10 = "PASS"
    elif ratio10 < 1.0:
        s10 = "WARN"
    else:
        s10 = "FAIL"
    note10 = f"sd_reps={sd_reps:.5f}, median_se={median_se:.5f}, ratio={ratio10:.3f}"
    checks["cross_fitting_stability"] = flag(s10, round(ratio10, 4), note10)
    print(f"Check 10  cross_fitting_stab.   : {s10}  ({note10})")
else:
    checks["cross_fitting_stability"] = flag("SKIP", None, "per_rep_coefs not available in Best learner")
    print(f"Check 10  cross_fitting_stab.   : SKIP  (no per_rep data)")

# ===================================================================
# Check 11: Learner sign agreement
# ===================================================================
learner_names = [k for k in primary["learners"] if k not in ("Ensemble", "Best")]
learner_coefs = {k: primary["learners"][k]["coef"] for k in learner_names}

if len(learner_coefs) > 1:
    signs = [np.sign(v) for v in learner_coefs.values()]
    n_pos = sum(1 for s in signs if s > 0)
    n_neg = sum(1 for s in signs if s < 0)
    total = n_pos + n_neg
    if n_pos == total or n_neg == total:
        s11 = "PASS"
        note11 = f"all {total} learners agree on sign (all positive)" if n_pos == total else f"all {total} learners agree on sign (all negative)"
    elif max(n_pos, n_neg) > total / 2:
        s11 = "WARN"
        note11 = f"{max(n_pos, n_neg)}/{total} agree; dissenters: " + ", ".join(
            k for k, v in learner_coefs.items() if np.sign(v) != np.sign(list(learner_coefs.values())[0])
        )
    else:
        s11 = "FAIL"
        note11 = f"learners split {n_pos} pos / {n_neg} neg — no robust direction"
    checks["learner_sign_agreement"] = flag(s11, {"n_pos": n_pos, "n_neg": n_neg}, note11)
    print(f"Check 11  learner_sign_agree.   : {s11}  ({note11})")
else:
    checks["learner_sign_agreement"] = flag("SKIP", None, "only one learner")
    print(f"Check 11  learner_sign_agree.   : SKIP")

# ===================================================================
# Check 12: Lasso variable selection
# ===================================================================
lasso = primary["learners"].get("Lasso", {})
lasso_diag = lasso.get("lasso_diagnostics")

if lasso_diag is not None:
    nn_out = lasso_diag.get("outcome_model", {}).get("n_nonzero", -1)
    nn_trt = lasso_diag.get("treatment_model", {}).get("n_nonzero", -1)
    if nn_out > 0 and nn_trt > 0:
        s12 = "PASS"
        note12 = f"outcome n_nonzero={nn_out}, treatment n_nonzero={nn_trt}"
    else:
        s12 = "WARN"
        degenerate = []
        if nn_out == 0:
            degenerate.append("outcome")
        if nn_trt == 0:
            degenerate.append("treatment")
        note12 = f"Lasso degenerate (predicting mean) for: {', '.join(degenerate)}"
    checks["lasso_variable_selection"] = flag(s12, {"n_nonzero_outcome": nn_out, "n_nonzero_treatment": nn_trt}, note12)
    print(f"Check 12  lasso_var_selection   : {s12}  ({note12})")
else:
    checks["lasso_variable_selection"] = flag("SKIP", None, "Lasso not among learners or lasso_diagnostics is null")
    print(f"Check 12  lasso_var_selection   : SKIP")

# ===================================================================
# Assemble output
# ===================================================================
active_statuses = [v["status"] for v in checks.values() if v["status"] != "SKIP"]
overall = "FAIL" if "FAIL" in active_statuses else ("WARN" if "WARN" in active_statuses else "PASS")
blocking = [k for k, v in checks.items() if v["status"] == "FAIL"]
warnings = [f"{k}: {v['note']}" for k, v in checks.items() if v["status"] == "WARN"]

diagnostics = {
    "overall": overall,
    "checks": checks,
    "blocking_issues": blocking,
    "warnings": warnings,
}

DIAGNOSTICS_FLAGS.write_text(json.dumps(diagnostics, indent=2))

print(f"\n{'='*60}")
print(f"OVERALL: {overall}")
print(f"{'='*60}")
if blocking:
    print(f"Blocking issues: {blocking}")
if warnings:
    print(f"Warnings: {warnings}")
print(f"\nWrote: {DIAGNOSTICS_FLAGS}")

Check  1  replication_pass      : WARN  (5.11%)
Check  2  dml_direction         : PASS  (same sign, 43.7% magnitude diff)
Check  3  dml_ci_coverage       : PASS  (published coef inside DML CI)
Check  4  nuisance_quality      : WARN  (r2_outcome=0.154, r2_treatment=0.362)
Check  5  sample_size           : PASS  (n=63)
Check  6  hte_heterogeneity     : PASS  (heterogeneity detected (non-overlapping GATE CIs))
Check  7  cf_ate_consistency    : SKIP  (no CF)
Check  8  cf_cate_plausibility  : SKIP  (no CF)
Check  9  ate_se_plausibility   : SKIP  (no CF)
Check 10  cross_fitting_stab.   : SKIP  (no per_rep data)
Check 11  learner_sign_agree.   : PASS  (all 5 learners agree on sign (all positive))
Check 12  lasso_var_selection   : PASS  (outcome n_nonzero=6, treatment n_nonzero=2)

OVERALL: WARN
Warnings: ['replication_pass: 5.11% max deviation', 'nuisance_quality: r2_outcome=0.154, r2_treatment=0.362']

Wrote: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\dat